# 🧪 A/B Test Impact Evaluation — Checkout Redesign
### Statistical & Business Decision Analysis

---

## 📋 Executive Summary

| Metric | Value |
|---|---|
| **Experiment** | Checkout Page Redesign |
| **Test Duration** | 2 weeks (simulated) |
| **Sample Size (per group)** | 3,835 users |
| **Control Conversion Rate** | 9.15% |
| **Treatment Conversion Rate** | 12.07% |
| **Relative Lift** | +31.9% |
| **p-value** | < 0.0001 |
| **Statistical Significance** | ✅ Yes |
| **Practical Significance** | ✅ Yes |
| **Estimated Monthly Revenue Uplift** | ₹2.33 Cr |
| **Final Recommendation** | 🚀 Ship |

---

## 1. Business Context & Problem Statement

### Background
An e-commerce platform was experiencing high cart abandonment rates (~90%) at the checkout stage. The product team hypothesized that a **redesigned checkout page** with a simplified UX flow would reduce friction and improve conversion.

### Business Question
> *Does the new checkout page design produce a statistically significant and practically meaningful improvement in conversion rate — enough to justify a full rollout?*

### Why This Matters
- Each 1% improvement in conversion ≈ ₹80L additional monthly revenue (at 1M monthly users, ₹800 AOV)
- Rolling out a bad change has real costs: engineering effort, user trust, future A/B test pollution
- We need both **statistical confidence** and **practical significance** before shipping

## 2. Experiment Design

| Parameter | Value | Rationale |
|---|---|---|
| **Randomization Unit** | User-level | Avoids network effects; ensures independence |
| **Control (A)** | Existing checkout page | Baseline behavior |
| **Treatment (B)** | Redesigned checkout page | New hypothesis |
| **Primary Metric** | Conversion Rate | Direct proxy for revenue |
| **Significance Level (α)** | 0.05 | Standard industry threshold |
| **Statistical Power (1-β)** | 80% | Probability of detecting true effect |
| **Minimum Detectable Effect** | 2% absolute lift (10% → 12%) | Smallest business-meaningful change |
| **Test Type** | Two-tailed Z-test for proportions | Binary outcome, large sample |

### Guardrail Metrics (monitored but not decision criteria)
- Page load time (no degradation in treatment)
- Error rate (no increase in checkout errors)
- Average order value (ensure uplift is real, not just cheap orders)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportions_ztest,
    proportion_confint,
    proportion_effectsize,
)

# ── Plot Style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'control': '#4C72B0', 'treatment': '#DD8452', 'neutral': '#55A868'}
sns.set_palette([COLORS['control'], COLORS['treatment']])

print("✅ Libraries loaded successfully")

## 3. Sample Size Calculation (Power Analysis)

Before running the experiment, we calculate the **minimum sample size** required to reliably detect our target effect. Running underpowered tests leads to false negatives; oversized tests waste resources and expose users unnecessarily.

**Formula parameters:**
- Baseline conversion rate: 10% (historical)
- Minimum detectable effect (MDE): 12% (2% absolute lift)
- α = 0.05 (Type I error)
- Power = 0.80 (Type II error = 0.20)

In [ ]:
# ── Experiment parameters ─────────────────────────────────────────────────────
baseline_conversion  = 0.10   # Control baseline
expected_conversion  = 0.12   # MDE: minimum detectable effect
alpha                = 0.05
power                = 0.80

# ── Effect size (Cohen's h for proportions) ───────────────────────────────────
effect_size = proportion_effectsize(baseline_conversion, expected_conversion)

# ── Required sample size per group ───────────────────────────────────────────
analysis = NormalIndPower()
required_n = int(np.ceil(analysis.solve_power(
    effect_size=effect_size, alpha=alpha, power=power
)))

print(f"{'='*50}")
print(f"  POWER ANALYSIS RESULTS")
print(f"{'='*50}")
print(f"  Baseline conversion rate : {baseline_conversion:.0%}")
print(f"  Expected conversion rate : {expected_conversion:.0%}")
print(f"  Cohen's h (effect size)  : {effect_size:.4f}")
print(f"  Significance level (α)   : {alpha}")
print(f"  Statistical power        : {power:.0%}")
print(f"  ──────────────────────────────────────────────")
print(f"  Required sample per group: {required_n:,}")
print(f"  Total users required     : {required_n*2:,}")
print(f"{'='*50}")

## 4. Data Simulation & Preparation

We simulate a realistic experiment dataset. In production, this data would come from your experimentation platform (e.g., Optimizely, LaunchDarkly, or a custom logging pipeline).

**Enrichments beyond the baseline:**
- Device type (mobile, desktop, tablet) — enables segmentation
- User type (new, returning) — enables cohort analysis
- Session duration — provides behavioral context

In [ ]:
np.random.seed(42)

total_users = required_n * 2
groups      = np.random.choice(['A', 'B'], size=total_users)

conv_probs = {'A': baseline_conversion, 'B': expected_conversion}

# ── Device & user type (adds realistic segmentation) ─────────────────────────
devices    = np.random.choice(['mobile', 'desktop', 'tablet'],
                               size=total_users, p=[0.55, 0.35, 0.10])
user_types = np.random.choice(['new', 'returning'],
                               size=total_users, p=[0.60, 0.40])

# ── Conversion (device-adjusted probabilities for realism) ───────────────────
device_multiplier = {'mobile': 0.85, 'desktop': 1.20, 'tablet': 1.00}

converted = [
    np.random.binomial(1, min(conv_probs[g] * device_multiplier[d], 1))
    for g, d in zip(groups, devices)
]

# ── Session duration (minutes, slightly higher for converted users) ───────────
session_duration = [
    round(np.random.exponential(8 if c == 1 else 4), 1)
    for c in converted
]

data = pd.DataFrame({
    'user_id'         : range(1, total_users + 1),
    'group'           : groups,
    'device'          : devices,
    'user_type'       : user_types,
    'converted'       : converted,
    'session_duration': session_duration,
})

print(f"Dataset shape: {data.shape}")
data.head(8)

## 5. Data Quality & Exploratory Analysis

Before any statistical testing, we validate the data integrity and understand the baseline distributions.

In [ ]:
# ── Data Quality Checks ───────────────────────────────────────────────────────
print("[1] Missing values:")
print(data.isnull().sum())

print("\n[2] Group distribution (should be ~50/50):")
group_counts = data['group'].value_counts()
print(group_counts)
print(f"    Split ratio: {group_counts['A']/total_users:.1%} / {group_counts['B']/total_users:.1%}")

print("\n[3] Duplicate users:")
print(f"    {data['user_id'].duplicated().sum()} duplicates found")

print("\n[4] Conversion summary:")
print(data.groupby('group')['converted'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'conversions', 'count': 'users', 'mean': 'conv_rate'}
).round(4))

In [ ]:
# ── EDA Visualizations ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exploratory Data Analysis — Experiment Overview', fontsize=14, fontweight='bold', y=1.02)

conv_rates = data.groupby('group')['converted'].mean()

# Plot 1: Conversion Rate Comparison
bars = axes[0].bar(['Control (A)', 'Treatment (B)'], conv_rates.values,
                   color=[COLORS['control'], COLORS['treatment']], width=0.5, edgecolor='white', linewidth=1.5)
for bar, rate in zip(bars, conv_rates.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{rate:.2%}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].set_title('Conversion Rate by Group', fontweight='bold')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_ylim(0, 0.18)
axes[0].axhline(y=baseline_conversion, color='grey', linestyle='--', alpha=0.6, label=f'Baseline ({baseline_conversion:.0%})')
axes[0].legend(fontsize=9)

# Plot 2: Device Distribution
device_conv = data.groupby(['device', 'group'])['converted'].mean().unstack()
device_conv.plot(kind='bar', ax=axes[1], color=[COLORS['control'], COLORS['treatment']],
                 width=0.6, edgecolor='white')
axes[1].set_title('Conversion Rate by Device', fontweight='bold')
axes[1].set_ylabel('Conversion Rate')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Control (A)', 'Treatment (B)'])

# Plot 3: User Type Breakdown
user_conv = data.groupby(['user_type', 'group'])['converted'].mean().unstack()
user_conv.plot(kind='bar', ax=axes[2], color=[COLORS['control'], COLORS['treatment']],
               width=0.5, edgecolor='white')
axes[2].set_title('Conversion Rate by User Type', fontweight='bold')
axes[2].set_ylabel('Conversion Rate')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(['Control (A)', 'Treatment (B)'])

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 EDA chart saved as eda_overview.png")

## 6. A/A Test Validation

**Why run an A/A test?**
Before trusting A/B results, we validate the experiment infrastructure by splitting the Control group in half and testing if we get a *false positive*. If the A/A test passes (p > 0.05), our randomization and measurement pipeline are working correctly.

⚠️ A failing A/A test would invalidate the entire experiment.

In [ ]:
# ── A/A Test ──────────────────────────────────────────────────────────────────
control = data[data['group'] == 'A'].copy()
control['aa_group'] = np.random.choice(['A1', 'A2'], size=len(control))

aa_success = control.groupby('aa_group')['converted'].sum()
aa_total   = control.groupby('aa_group')['converted'].count()

aa_z, aa_p = proportions_ztest(aa_success, aa_total)
aa_rates   = control.groupby('aa_group')['converted'].mean()

print(f"{'='*50}")
print(f"  A/A TEST RESULTS")
print(f"{'='*50}")
print(f"  A1 conversion rate : {aa_rates['A1']:.4f}")
print(f"  A2 conversion rate : {aa_rates['A2']:.4f}")
print(f"  Z-statistic        : {aa_z:.4f}")
print(f"  p-value            : {aa_p:.4f}")
print(f"  ──────────────────────────────────────────────")

if aa_p > alpha:
    print(f"  ✅ PASS — No significant difference (p={aa_p:.3f} > α={alpha})")
    print(f"     Randomization is valid. Proceed with A/B test.")
else:
    print(f"  ❌ FAIL — Significant difference detected (p={aa_p:.3f} < α={alpha})")
    print(f"     ⚠️  Experiment infrastructure may be flawed.")
print(f"{'='*50}")

## 7. Hypothesis Testing

### Hypotheses

| | Hypothesis |
|---|---|
| **H₀** (Null) | p_B − p_A = 0 (no difference in conversion rate) |
| **H₁** (Alternative) | p_B − p_A ≠ 0 (two-tailed; treatment may help or hurt) |

### Assumptions Check
- ✅ **Independence**: Users randomly assigned; no cross-contamination
- ✅ **Normality**: Binary outcome + large n → Central Limit Theorem applies
- ✅ **Equal variance**: Proportion-based test accounts for this
- ✅ **A/A test**: Passed (Section 6)

In [ ]:
# ── Z-Test for Proportions ────────────────────────────────────────────────────
success_counts = data.groupby('group')['converted'].sum()
sample_counts  = data.groupby('group')['converted'].count()
conv_rates     = data.groupby('group')['converted'].mean()

z_stat, p_value = proportions_ztest(
    count=success_counts,
    nobs=sample_counts
)

# Cohen's h effect size
cohens_h = proportion_effectsize(conv_rates['B'], conv_rates['A'])

# Effect size interpretation
if abs(cohens_h) < 0.2:
    effect_label = "Small"
elif abs(cohens_h) < 0.5:
    effect_label = "Medium"
else:
    effect_label = "Large"

print(f"{'='*55}")
print(f"  HYPOTHESIS TEST RESULTS")
print(f"{'='*55}")
print(f"  Control (A) conversions : {success_counts['A']:,} / {sample_counts['A']:,} = {conv_rates['A']:.4f}")
print(f"  Treatment (B) conversions: {success_counts['B']:,} / {sample_counts['B']:,} = {conv_rates['B']:.4f}")
print(f"  ──────────────────────────────────────────────────")
print(f"  Z-statistic              : {z_stat:.4f}")
print(f"  p-value                  : {p_value:.6f}")
print(f"  Cohen's h (effect size)  : {cohens_h:.4f} ({effect_label})")
print(f"  ──────────────────────────────────────────────────")

if p_value < alpha:
    print(f"  ✅ REJECT H₀ — p={p_value:.5f} < α={alpha}")
    print(f"     Statistically significant difference detected.")
else:
    print(f"  ❌ FAIL TO REJECT H₀ — p={p_value:.5f} ≥ α={alpha}")
    print(f"     No significant difference detected.")
print(f"{'='*55}")

## 8. Confidence Intervals & Lift Analysis

In [ ]:
# ── Confidence Intervals (Wilson method — better for proportions) ─────────────
ci_A = proportion_confint(success_counts['A'], sample_counts['A'], alpha=alpha, method='wilson')
ci_B = proportion_confint(success_counts['B'], sample_counts['B'], alpha=alpha, method='wilson')

# ── Lift ──────────────────────────────────────────────────────────────────────
absolute_lift = conv_rates['B'] - conv_rates['A']
relative_lift = absolute_lift / conv_rates['A']

# ── Lift Confidence Interval (conservative: subtract overlapping bounds) ──────
lift_ci_low  = ci_B[0] - ci_A[1]
lift_ci_high = ci_B[1] - ci_A[0]

print(f"{'='*60}")
print(f"  CONFIDENCE INTERVALS (95%, Wilson method)")
print(f"{'='*60}")
print(f"  Control (A)  : [{ci_A[0]:.4f}, {ci_A[1]:.4f}]  → point est: {conv_rates['A']:.4f}")
print(f"  Treatment (B): [{ci_B[0]:.4f}, {ci_B[1]:.4f}]  → point est: {conv_rates['B']:.4f}")
print(f"  ──────────────────────────────────────────────────────")
print(f"  Absolute lift: {absolute_lift:.4f}  ({absolute_lift:.2%})")
print(f"  Relative lift: {relative_lift:.4f}  ({relative_lift:.2%})")
print(f"  Lift 95% CI  : [{lift_ci_low:.4f}, {lift_ci_high:.4f}]")
print()

if lift_ci_low > 0:
    print(f"  ✅ Lift CI entirely above 0 — treatment is reliably better")
else:
    print(f"  ⚠️  Lift CI crosses 0 — treatment effect is uncertain")
print(f"{'='*60}")

In [ ]:
# ── Confidence Interval Visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Statistical Significance — Confidence Interval Analysis', fontsize=13, fontweight='bold')

# Left: CI overlap plot
groups_plot = ['Control (A)', 'Treatment (B)']
means  = [conv_rates['A'], conv_rates['B']]
errors = [
    [conv_rates['A'] - ci_A[0], ci_A[1] - conv_rates['A']],
    [conv_rates['B'] - ci_B[0], ci_B[1] - conv_rates['B']],
]

for i, (g, m, e, c) in enumerate(zip(groups_plot, means, errors, [COLORS['control'], COLORS['treatment']])):
    axes[0].errorbar(i, m, yerr=[[e[0]], [e[1]]], fmt='o', color=c,
                     capsize=8, capthick=2, elinewidth=2, markersize=10, label=g)
    axes[0].axhspan(ci_A[0], ci_A[1], alpha=0.08, color=COLORS['control'])

axes[0].set_title('Conversion Rate with 95% Confidence Intervals', fontweight='bold')
axes[0].set_ylabel('Conversion Rate')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(groups_plot)
axes[0].set_xlim(-0.5, 1.5)
axes[0].legend()
axes[0].annotate(f'Lift: +{absolute_lift:.2%}\n(p < 0.0001)',
                 xy=(0.5, (conv_rates['A'] + conv_rates['B']) / 2),
                 ha='center', fontsize=10, color='darkgreen', fontweight='bold')

# Right: Lift distribution (bootstrap)
n_boot = 5000
boot_lifts = []
for _ in range(n_boot):
    boot_a = np.random.binomial(sample_counts['A'], conv_rates['A']) / sample_counts['A']
    boot_b = np.random.binomial(sample_counts['B'], conv_rates['B']) / sample_counts['B']
    boot_lifts.append(boot_b - boot_a)

axes[1].hist(boot_lifts, bins=50, color=COLORS['treatment'], alpha=0.7, edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='No effect (H₀)')
axes[1].axvline(x=absolute_lift, color='darkgreen', linewidth=2,
                label=f'Observed lift ({absolute_lift:.2%})')
pct_positive = np.mean(np.array(boot_lifts) > 0) * 100
axes[1].set_title(f'Bootstrap Distribution of Lift\n({pct_positive:.1f}% of bootstraps show positive lift)', fontweight='bold')
axes[1].set_xlabel('Absolute Lift (B − A)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('confidence_intervals.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 CI chart saved as confidence_intervals.png")

## 9. Segment Analysis

**Why segment?**
A positive overall result can mask heterogeneous effects. A treatment that works great on desktop may hurt mobile users. Segmentation helps us:
1. Validate robustness of the effect
2. Identify which user groups drive the lift
3. Inform rollout strategy (e.g., roll out to desktop first)

In [ ]:
# ── Segment Analysis: Device × Group ─────────────────────────────────────────
segments = []

for segment_col in ['device', 'user_type']:
    for seg_val in data[segment_col].unique():
        seg_data = data[data[segment_col] == seg_val]
        seg_success = seg_data.groupby('group')['converted'].sum()
        seg_total   = seg_data.groupby('group')['converted'].count()
        seg_rates   = seg_data.groupby('group')['converted'].mean()

        if 'A' in seg_success.index and 'B' in seg_success.index:
            z, p = proportions_ztest(
                count=[seg_success['A'], seg_success['B']],
                nobs=[seg_total['A'], seg_total['B']]
            )
            lift = seg_rates['B'] - seg_rates['A']
            rel_lift = lift / seg_rates['A']

            segments.append({
                'Segment Type': segment_col,
                'Segment'     : seg_val,
                'n (Control)' : seg_total.get('A', 0),
                'n (Treatment)': seg_total.get('B', 0),
                'Conv (A)'    : f"{seg_rates['A']:.3%}",
                'Conv (B)'    : f"{seg_rates['B']:.3%}",
                'Abs Lift'    : f"{lift:.3%}",
                'Rel Lift'    : f"{rel_lift:.1%}",
                'p-value'     : round(p, 4),
                'Significant?': '✅' if p < alpha else '❌'
            })

seg_df = pd.DataFrame(segments)
print("SEGMENT ANALYSIS RESULTS")
print("="*80)
print(seg_df.to_string(index=False))

In [ ]:
# ── Segment Lift Visualisation ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Segment Analysis — Treatment Effect Across Groups', fontsize=13, fontweight='bold')

for ax, segment_col in zip(axes, ['device', 'user_type']):
    seg_subset = seg_df[seg_df['Segment Type'] == segment_col].copy()
    seg_subset['abs_lift_float'] = [
        float(x.replace('%', '')) / 100 for x in seg_subset['Abs Lift']
    ]
    colors = ['#55A868' if v > 0 else '#C44E52' for v in seg_subset['abs_lift_float']]

    bars = ax.barh(seg_subset['Segment'], seg_subset['abs_lift_float'],
                   color=colors, edgecolor='white', linewidth=1)
    ax.axvline(x=0, color='black', linewidth=0.8)
    ax.axvline(x=absolute_lift, color='darkblue', linestyle='--',
               linewidth=1.5, label=f'Overall lift ({absolute_lift:.2%})')

    for bar, val in zip(bars, seg_subset['abs_lift_float']):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.2%}', va='center', fontsize=9, fontweight='bold')

    ax.set_title(f'Lift by {segment_col.title()}', fontweight='bold')
    ax.set_xlabel('Absolute Lift (Treatment − Control)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Segment chart saved as segment_analysis.png")

## 10. Business Impact Analysis

Statistical significance is necessary but not sufficient. We must translate the result into **₹ revenue impact** and run sensitivity analysis to understand the range of outcomes.

In [ ]:
# ── Core Business Impact ──────────────────────────────────────────────────────
monthly_users       = 1_000_000
avg_order_value     = 800       # INR

monthly_uplift = absolute_lift * monthly_users * avg_order_value
annual_uplift  = monthly_uplift * 12

current_monthly_rev = conv_rates['A'] * monthly_users * avg_order_value
new_monthly_rev     = conv_rates['B'] * monthly_users * avg_order_value

print(f"{'='*60}")
print(f"  BUSINESS IMPACT SUMMARY")
print(f"{'='*60}")
print(f"  Monthly active users    : {monthly_users:>12,}")
print(f"  Avg order value (INR)   : {avg_order_value:>12,}")
print(f"  ──────────────────────────────────────────────────────")
print(f"  Control conversion rate : {conv_rates['A']:>12.2%}")
print(f"  Treatment conversion rate:{conv_rates['B']:>12.2%}")
print(f"  Absolute lift           : {absolute_lift:>12.2%}")
print(f"  Relative lift           : {relative_lift:>12.2%}")
print(f"  ──────────────────────────────────────────────────────")
print(f"  Current monthly revenue : ₹{current_monthly_rev:>10,.0f}")
print(f"  Projected monthly revenue: ₹{new_monthly_rev:>10,.0f}")
print(f"  ──────────────────────────────────────────────────────")
print(f"  📈 Monthly uplift       : ₹{monthly_uplift:>10,.0f}  ({monthly_uplift/1e7:.2f} Cr)")
print(f"  📈 Annual uplift        : ₹{annual_uplift:>10,.0f}  ({annual_uplift/1e7:.2f} Cr)")
print(f"{'='*60}")

In [ ]:
# ── Sensitivity Analysis ──────────────────────────────────────────────────────
# What if actual lift is at the lower end of the confidence interval?

scenarios = {
    'Conservative (CI lower bound)': lift_ci_low,
    'Point Estimate (observed lift)': absolute_lift,
    'Optimistic (CI upper bound)'  : lift_ci_high,
}

print(f"{'='*65}")
print(f"  SENSITIVITY ANALYSIS — Revenue Uplift Scenarios")
print(f"{'='*65}")
print(f"  {'Scenario':<38} {'Lift':>8}  {'Monthly':>12}  {'Annual':>12}")
print(f"  {'-'*63}")
for name, lift in scenarios.items():
    m_rev = lift * monthly_users * avg_order_value
    a_rev = m_rev * 12
    print(f"  {name:<38} {lift:>7.2%}  ₹{m_rev:>10,.0f}  ₹{a_rev:>10,.0f}")
print(f"{'='*65}")

In [ ]:
# ── Sensitivity Visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Business Impact Analysis', fontsize=13, fontweight='bold')

# Left: Revenue scenarios
scenario_names  = ['Conservative', 'Point Estimate', 'Optimistic']
monthly_impacts = [lift * monthly_users * avg_order_value / 1e6 for lift in scenarios.values()]
sc_colors = ['#4C72B0', '#55A868', '#DD8452']

bars = axes[0].bar(scenario_names, monthly_impacts, color=sc_colors, width=0.5, edgecolor='white')
for bar, val in zip(bars, monthly_impacts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'₹{val:.1f}M', ha='center', fontweight='bold', fontsize=10)
axes[0].set_title('Monthly Revenue Uplift — Scenarios (₹ Million)', fontweight='bold')
axes[0].set_ylabel('Revenue Uplift (₹ Millions)')
axes[0].set_ylim(0, max(monthly_impacts) * 1.25)

# Right: Lift vs AOV heatmap (how does revenue change across different scenarios)
aov_range  = np.arange(400, 1601, 200)
lift_range = np.arange(0.01, 0.06, 0.01)
revenue_matrix = np.array([
    [l * monthly_users * a / 1e6 for a in aov_range]
    for l in lift_range
])

im = axes[1].imshow(revenue_matrix, cmap='YlGn', aspect='auto')
axes[1].set_xticks(range(len(aov_range)))
axes[1].set_xticklabels([f'₹{a}' for a in aov_range])
axes[1].set_yticks(range(len(lift_range)))
axes[1].set_yticklabels([f'{l:.0%}' for l in lift_range])
axes[1].set_xlabel('Average Order Value (INR)')
axes[1].set_ylabel('Absolute Lift')
axes[1].set_title('Monthly Revenue Uplift Heatmap (₹ Millions)', fontweight='bold')

for i in range(len(lift_range)):
    for j in range(len(aov_range)):
        axes[1].text(j, i, f'{revenue_matrix[i, j]:.0f}', ha='center', va='center',
                     fontsize=8, color='black' if revenue_matrix[i, j] < 40 else 'white')

plt.colorbar(im, ax=axes[1], label='₹ Millions')
plt.tight_layout()
plt.savefig('business_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Business impact chart saved as business_impact.png")

## 11. Decision Framework & Final Recommendation

In [ ]:
# ── Decision Criteria ─────────────────────────────────────────────────────────
min_practical_lift = 0.01   # 1% absolute = business threshold

criteria = {
    'Statistical significance (p < 0.05)'    : p_value < alpha,
    'Practical significance (lift ≥ 1%)'     : absolute_lift >= min_practical_lift,
    'A/A test passed'                         : aa_p > alpha,
    'Lift CI entirely above zero'             : lift_ci_low > 0,
    'Effect consistent across device segments': all([
        float(r['Abs Lift'].replace('%', '')) > 0
        for _, r in seg_df[seg_df['Segment Type'] == 'device'].iterrows()
    ]),
}

print(f"{'='*60}")
print(f"  DECISION FRAMEWORK CHECKLIST")
print(f"{'='*60}")
all_pass = True
for criterion, passed in criteria.items():
    icon = '✅' if passed else '❌'
    print(f"  {icon}  {criterion}")
    if not passed:
        all_pass = False
print(f"{'='*60}")
print()

if all_pass:
    print("  🚀 FINAL RECOMMENDATION: SHIP")
    print()
    print("  The redesigned checkout page has demonstrated:")
    print(f"  • +{absolute_lift:.2%} absolute lift in conversion rate")
    print(f"  • +{relative_lift:.1%} relative improvement over baseline")
    print(f"  • p-value = {p_value:.5f} (strong statistical significance)")
    print(f"  • Estimated monthly revenue uplift: ₹{monthly_uplift/1e7:.2f} Cr")
    print()
    print("  Recommended next steps:")
    print("  1. Gradual rollout: 10% → 25% → 50% → 100% traffic")
    print("  2. Monitor guardrail metrics (error rate, page load, AOV)")
    print("  3. Schedule post-launch review at Day 7 and Day 30")
    print("  4. Run follow-up segmented test on mobile (lower lift observed)")
else:
    print("  ⛔ FINAL RECOMMENDATION: DO NOT SHIP")
    print("  Not all decision criteria were met. Review failed checks above.")

print(f"{'='*60}")

## 12. Limitations & Future Work

### Current Limitations

| Limitation | Risk | Mitigation |
|---|---|---|
| **Simulated data** | May not capture real user behaviour | Validate on live traffic with holdout |
| **Novelty effect** | Users may respond to newness, not improvement | Re-test after 4+ weeks |
| **Short experiment window** | Seasonal effects not captured | Run across multiple weeks/months |
| **Single primary metric** | Optimising one metric may hurt others | Add guardrail metrics to framework |
| **No network effects modelled** | Social referrals not captured | Use switchback or cluster randomisation |

### Future Work

1. **Bayesian A/B Testing** — Provides probability of being best (PROB) instead of binary p-value; better for early stopping decisions
2. **Sequential Testing** (SPRT / always-valid inference) — Allows peeking at results without inflating Type I error
3. **Multi-armed Bandit** — Dynamically shifts traffic to winning variant; maximises revenue during experiment
4. **Uplift Modelling / CATE** — Estimate heterogeneous treatment effects: which individual users benefit most?
5. **Long-term retention analysis** — Does the checkout improvement affect 30/60/90-day retention?

---

*Notebook by Mahasweta Talik — A/B Test Impact Evaluation*